# Exploring Ethereum mainnet on BigQuery

Dataset: `bigquery-public-data.goog_blockchain_ethereum_mainnet_us`

**The cost model in three sentences.** BigQuery charges by bytes *scanned*
(first 1 TB per month is free), and `LIMIT` does **not** reduce scanning.
Cost is controlled by selecting few columns and filtering on the partition
column `block_timestamp` (all tables here are partitioned by month on it).
`dry_run()` shows what a query *would* scan for free, and `run_query()`
refuses anything that would bill more than `max_gb` (default 10 GB).

In [1]:
import pandas as pd

from eth_graph_research.bq import DATASET, dry_run, run_query

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
DATASET

'bigquery-public-data.goog_blockchain_ethereum_mainnet_us'

## What tables exist?

`INFORMATION_SCHEMA` queries are metadata-only and cost next to nothing.

In [2]:
run_query(f'''
SELECT table_name
FROM `{DATASET}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
''')

Scanned 0.010 GB (billed 0.010 GB)


,table_name
0,accounts
1,accounts_state
2,accounts_state_by_address
3,blocks
4,decoded_events
5,logs
6,receipts
7,token_transfers
8,traces
9,transactions


## What columns do the key tables have?

Keep this DataFrame handy — it is the ground truth for column names and
types when you write your own queries.

In [3]:
schema = run_query(f'''
SELECT table_name, column_name, data_type
FROM `{DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('blocks', 'transactions', 'token_transfers')
ORDER BY table_name, ordinal_position
''')
schema

Scanned 0.010 GB (billed 0.010 GB)


,table_name,column_name,data_type
0,blocks,block_hash,STRING
1,blocks,block_number,INT64
2,blocks,block_timestamp,TIMESTAMP
3,blocks,parent_hash,STRING
4,blocks,size,INT64
5,blocks,extra_data,STRING
6,blocks,gas_limit,INT64
7,blocks,gas_used,INT64
8,blocks,base_fee_per_gas,INT64
9,blocks,mix_hash,STRING


## Dry runs: check cost before you spend it

Three estimates for the `transactions` table:
1. `SELECT *`, no filter — scans the entire table (terabytes!).
2. Same with `LIMIT 10` — **identical cost**. LIMIT is applied after scanning.
3. Two columns + one month of `block_timestamp` — a tiny fraction.

Rule of thumb: never run a query against a big table without a
`block_timestamp` filter, and always `dry_run` anything new.

In [4]:
dry_run(f'SELECT * FROM `{DATASET}.transactions`')
dry_run(f'SELECT * FROM `{DATASET}.transactions` LIMIT 10')
dry_run(f'''
SELECT transaction_hash, from_address
FROM `{DATASET}.transactions`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
''')

Dry run: would scan 4,229.857 GB


Dry run: would scan 4,229.857 GB


Dry run: would scan 8.543 GB


8.5430232

## Example 1 — blocks from the last day

`blocks` is a small table; a day of it is cheap. Partitions are monthly, so
a 1-day filter still scans the whole current month's partition for the
selected columns — that's fine here.

In [5]:
run_query(f'''
SELECT
  COUNT(*)              AS n_blocks,
  MIN(block_number)     AS first_block,
  MAX(block_number)     AS last_block,
  ROUND(AVG(gas_used))  AS avg_gas_used
FROM `{DATASET}.blocks`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
''')

Scanned 0.001 GB (billed 0.010 GB)


,n_blocks,first_block,last_block,avg_gas_used
0,7084,25634873,25641956,30365438.0


## Example 2 — transactions per day, last 7 days

The same shape as Google's own example query for this dataset.

In [6]:
run_query(f'''
SELECT
  TIMESTAMP_TRUNC(block_timestamp, DAY) AS day,
  COUNT(*) AS txn_count
FROM `{DATASET}.transactions`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY day
ORDER BY day
''')

Scanned 0.110 GB (billed 0.110 GB)


,day,txn_count
0,2026-07-23 00:00:00+00:00,2324696
1,2026-07-24 00:00:00+00:00,2552428
2,2026-07-25 00:00:00+00:00,1998222
3,2026-07-26 00:00:00+00:00,1538710
4,2026-07-27 00:00:00+00:00,1871418
5,2026-07-28 00:00:00+00:00,1664214
6,2026-07-29 00:00:00+00:00,1734060
7,2026-07-30 00:00:00+00:00,4106


## Example 3 — a peek at recent individual transactions

Selecting few columns keeps the scan small; `value` is in wei
(1 ETH = 1e18 wei). The LIMIT here is for display only — the
`block_timestamp` filter is what keeps it cheap.

In [7]:
run_query(f'''
SELECT
  block_timestamp,
  transaction_hash,
  from_address,
  to_address,
  SAFE_CAST(value AS FLOAT64) / 1e18 AS value_eth
FROM `{DATASET}.transactions`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
ORDER BY block_timestamp DESC
LIMIT 20
''')

Scanned 0.010 GB (billed 0.010 GB)


,block_timestamp,transaction_hash,from_address,to_address,value_eth
0,2026-07-30 00:03:35+00:00,0x8e3e2ad322928b9288549062b5e901745400ba1bb270...,0x22b338deac679d611586812303427f6369eae519,0xd152f549545093347a162dce210e7293f1452150,0.000132
1,2026-07-30 00:03:35+00:00,0x71ce54c543283295eeb7c422d237fb8c4f3f89818135...,0x2252f216f4a494a87025123425181ca1bb754fb8,0x0000000aa232009084bd71a5797d089aa4edfad4,0.000000
2,2026-07-30 00:03:35+00:00,0xd2e5ea63f640a01e9d49bd2d7f803adafa62f0f28e85...,0x621101f4ed4aa4af48c5baf13638a41c7e546020,0xeff6cb8b614999d130e537751ee99724d01aa167,0.000000
3,2026-07-30 00:03:35+00:00,0x1efbb6354830d82d4d0e195c0b313358f381dcd7f5d8...,0x8b5606469e21c75edd7c74d1c7a65824675d9aff,0x9ccc2f3ecde026230e11a5c8799ac7524f2bb294,0.000000
4,2026-07-30 00:03:35+00:00,0x4898fa05838f2c126bd4e554d7e8ce35f9f31f8e9603...,0x883b07932de27d7f679ea1b2804366cfd10151eb,0x3328f7f4a1d1c57c35df56bbf0c9dcafca309c49,0.049700
5,2026-07-30 00:03:35+00:00,0xac9159e18a68ec936f002275f1703dcec47e66a77b7e...,0xb42f812a44c22cc6b861478900401ee759ebead6,0x9eab1a0f065366e215778ea0d9a60f2207f6b301,0.000242
6,2026-07-30 00:03:35+00:00,0xffebf6f90346b3d0d63a36a0cd95e5b6052026ed5376...,0x05ff6964d21e5dae3b1010d5ae0465b3c450f381,0x2744dfd9898f0babbc570cc594bbbc84b487a22b,0.000000
7,2026-07-30 00:03:35+00:00,0x575342a51357d76996418d28c9ca426687eb14d47edd...,0x6745e16220971949ae43c1dfc2a13573d4329395,0xe60fae7817aeee84f378cde1ea2587795694347e,0.000000
8,2026-07-30 00:03:35+00:00,0xc674ee6fa6be8b2266d1b1c3f75a3faac0d335dd78a8...,0x158c6c80b579cdab55be91d1b176f56fce489b92,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.030000
9,2026-07-30 00:03:35+00:00,0xbd107cdc3203a1bd7500f08fd0d1dba0aca211fabe51...,0x8c31093e57f9827c162f8f190ee29669f8bf2ee9,0x66a9893cc07d91d95644aedd05d03f95e1dba8af,0.000000


## Example 4 — busiest token contracts in the last day

`token_transfers` is decoded ERC-20 `Transfer` events. `address` is the
token contract emitting the event. (If the schema cell above shows a
different column name, use that.)

In [8]:
run_query(f'''
SELECT
  address AS token_contract,
  COUNT(*) AS transfers
FROM `{DATASET}.token_transfers`
WHERE block_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
GROUP BY token_contract
ORDER BY transfers DESC
LIMIT 10
''')

Scanned 0.185 GB (billed 0.186 GB)


,token_contract,transfers
0,0xdac17f958d2ee523a2206206994597c13d831ec7,984861
1,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,699362
2,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,263426
3,0xc80b4457fccd49e7c3a2f5566dd1b827d85f6c98,165137
4,0xbeef007ecfbfdf9b919d0050821a9b6dbd634ff0,87298
5,0x647a139b234dcf9f91b1b749993604e715d3acb8,67757
6,0xaa8a56638b9f91fffa3188693731a8fcbcf40a3b,54618
7,0x2831b7136c49c23f53e3da38693fe00af1481b2a,40188
8,0x1aed8c8e8f5ac86800b1e914d797929f0f93f9c1,35031
9,0x56dff0942be7e6f3300279301776bb25ec7858fc,31336


## Where next

- Cross-reference token contracts against `crypto_ethereum.tokens`
  (the community dataset) or `decoded_events` to get symbols/decimals.
- For graph research: `transactions` (`from_address` → `to_address`) and
  `token_transfers` are the edge lists. Decide on a time window, dry-run
  the extraction query, then export.
- Raise `max_gb` per call only when a dry run justifies it:
  `run_query(sql, max_gb=50)`.